# Pokémon Battle Prediction 2026 — Kaggle Submission Generator & Validator

Questo notebook è progettato specificamente per l'ambiente **Kaggle (CPU o GPU)** con un unico obiettivo chiaro:
1. **Eseguire la pipeline completa di inferenza/ensemble** (5-Fold Stratified CV, 4 architetture XGBoost complementari, Zero-Leakage Within-Fold Target Encoding e blending SLSQP robusto).
2. **Generare il file `submission.csv`** direttamente nella cartella di lavoro `/kaggle/working/submission.csv` (o locale `submission.csv`).
3. **Eseguire un controllo rigoroso (Sanity & Integrity Check)** che certifica che il file è pronto e conforme al 100% per l'invio sul sito di Kaggle.

## Step 1: Bootstrap Ambiente e Repository GitHub
Rileviamo se siamo su Kaggle e cloniamo il repository aggiornato in `/tmp/predictive-spikes` (in modo da non sporcare `/kaggle/working/` con file non ammessi da Kaggle).

In [ ]:
# Standard library imports for filesystem navigation and process execution
import os
import shutil
import subprocess
import sys
import time
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
os.environ["PYTHONHASHSEED"] = "42"

REPO_URL = os.environ.get("REPO_URL", "https://github.com/danielegabriele17-rgb/predictive-spikes.git")
GIT_REF = os.environ.get("GIT_REF", "main")
CLONE_DIR = Path("/tmp/predictive-spikes")

def setup_repository() -> Path:
    """Locate the repository root, inspect Kaggle inputs, or clone cleanly."""
    # 1. Controlla cartelle locali o dataset Kaggle collegati
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        CLONE_DIR,
        Path("/kaggle/input/predictive-spikes"),
        Path("/kaggle/input/predictive-spikes-code"),
    ]
    for candidate in candidates:
        if (candidate / "src" / "battle_features.py").exists():
            print(f"Repository rilevato in: {candidate.resolve()}")
            return candidate.resolve()

    # 2. Se CLONE_DIR esiste già ma è vuota o corrotta, ripuliscila prima del clone
    if CLONE_DIR.exists():
        print(f"Directory {CLONE_DIR} esistente trovata. Rimozione per clone pulito...")
        shutil.rmtree(CLONE_DIR, ignore_errors=True)

    print(f"Clonazione di {REPO_URL} in {CLONE_DIR}...")
    try:
        res = subprocess.run(
            ["git", "clone", "--depth", "1", "-b", GIT_REF, REPO_URL, str(CLONE_DIR)],
            capture_output=True,
            text=True,
            check=True,
        )
        print("Clone completato con successo.")
        return CLONE_DIR.resolve()
    except subprocess.CalledProcessError as e:
        print("\n" + "!" * 75)
        print("ERRORE DURANTE IL GIT CLONE (Exit code 128):")
        print("Standard error:", e.stderr.strip() if e.stderr else "N/A")
        print("Standard output:", e.stdout.strip() if e.stdout else "N/A")
        print("!" * 75)
        if "Could not resolve host" in str(e.stderr) or "Connection refused" in str(e.stderr):
            print("\n>>> SOLUZIONE: In Kaggle, assicurati che 'Internet' sia attivato nel pannello laterale:")
            print("    Notebook Settings (a destra) -> 'Internet: On'.")
        raise e

# Setup repository root e sys.path
REPO_ROOT = setup_repository()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Repository Root : {REPO_ROOT}")
print(f"Working Dir     : {Path.cwd().resolve()}")


## Step 2: Caricamento Dati della Competizione
Carichiamo `train.jsonl`, `test.jsonl`, `sample_submission.csv` e la type chart di 2ª Generazione.

In [ ]:
import numpy as np
import pandas as pd
from src.data_loader import discover_data_dir, load_competition_data

data_dir = discover_data_dir()
train_battles, test_battles, sample_sub, type_chart = load_competition_data(data_dir)
y = np.array([int(b["player_won"]) for b in train_battles], dtype=int)

print(f"Directory dati        : {data_dir}")
print(f"Battaglie Training    : {len(train_battles):,}")
print(f"Battaglie Test        : {len(test_battles):,}")
print(f"Coppie Type Chart GSC : {len(type_chart)}")

## Step 3: Estrazione Feature Matrix e Pokédex Interno
Costruiamo il catalogo deterministico delle specie dal training set e ricaviamo le feature fisiche dai primi 30 turni.

In [ ]:
import time
from src.battle_features import build_species_catalogue
from src.feature_engineering import ALL_FEATURE_FAMILIES, build_feature_dataframe

t0 = time.perf_counter()
catalogue = build_species_catalogue(train_battles)
print(f"Catalogo specie registrate: {len(catalogue)}")

X_train_full, _, _ = build_feature_dataframe(
    train_battles, families=ALL_FEATURE_FAMILIES, catalogue=catalogue, type_chart=type_chart
)
X_test_full, test_ids, _ = build_feature_dataframe(
    test_battles, families=ALL_FEATURE_FAMILIES, catalogue=catalogue, type_chart=type_chart
)

print(f"Estrazione completata in {time.perf_counter() - t0:.2f}s")
print(f"  X_train_full : {X_train_full.shape}")
print(f"  X_test_full  : {X_test_full.shape}")

## Step 4: Addestramento Ensemble a 4 Modelli (5-Fold Stratified CV)
Addestriamo le 4 configurazioni XGBoost strutturalmente diversificate con Target Encoding calcolato rigorosamente entro ciascun fold.

In [ ]:
from typing import Dict, List
from src.validation import get_cv_splits
from src.feature_engineering import WithinFoldTargetEncoder, get_default_feature_subsets
from src.model import run_xgb_cv

RANDOM_SEED = 42
FINAL_FOLDS = 5
DECISION_THRESHOLD = 0.50

subsets = get_default_feature_subsets()
cv_splits = get_cv_splits(train_battles, y, n_splits=FINAL_FOLDS, seed=RANDOM_SEED, group_duplicates=False)

# Precomputazione Target Encoding rigorosamente within-fold (zero leakage)
te_folds_train: List[pd.DataFrame] = []
te_folds_val: List[pd.DataFrame] = []
te_folds_test: List[pd.DataFrame] = []
for tr_idx, va_idx in cv_splits:
    enc = WithinFoldTargetEncoder(smoothing=30.0, team_smoothing=20.0)
    tr_b = [train_battles[i] for i in tr_idx]
    va_b = [train_battles[i] for i in va_idx]
    te_folds_train.append(enc.fit_transform_inner_oof(tr_b, y[tr_idx], n_inner_splits=5, seed=RANDOM_SEED))
    te_folds_val.append(enc.transform(va_b))
    te_folds_test.append(enc.transform(test_battles))

fold_map = {(int(tr[0]), int(va[0])): idx for idx, (tr, va) in enumerate(cv_splits)}

def fold_te_fn(tr_idx: np.ndarray, va_idx: np.ndarray):
    idx = fold_map[(int(tr_idx[0]), int(va_idx[0]))]
    return te_folds_train[idx], te_folds_val[idx]

def test_te_fn(tr_idx: np.ndarray, va_idx: np.ndarray):
    idx = fold_map[(int(tr_idx[0]), int(va_idx[0]))]
    return te_folds_test[idx]

models_cfg = {
    "model1_optuna_top200_te": {
        "subset": "top200_features",
        "use_te": True,
        "params": {
            "n_estimators": 2500, "learning_rate": 0.023502, "max_depth": 4,
            "min_child_weight": 14, "subsample": 0.784829, "colsample_bytree": 0.556746,
            "colsample_bylevel": 0.779307, "reg_alpha": 1.547147, "reg_lambda": 6.103097, "gamma": 0.382765,
        },
    },
    "model2_conservative_depth4_pruned_te": {
        "subset": "pruned_features",
        "use_te": True,
        "params": {
            "n_estimators": 2500, "learning_rate": 0.02, "max_depth": 4,
            "min_child_weight": 18, "subsample": 0.75, "colsample_bytree": 0.55,
            "colsample_bylevel": 1.0, "reg_alpha": 2.0, "reg_lambda": 8.0, "gamma": 0.30,
        },
    },
    "model3_obs_only_top220_no_te": {
        "subset": "top220_features",
        "use_te": False,
        "params": {
            "n_estimators": 2500, "learning_rate": 0.02, "max_depth": 5,
            "min_child_weight": 12, "subsample": 0.80, "colsample_bytree": 0.60,
            "colsample_bylevel": 1.0, "reg_alpha": 1.5, "reg_lambda": 5.0, "gamma": 0.25,
        },
    },
    "model4_compact_top140_te": {
        "subset": "top140_features",
        "use_te": True,
        "params": {
            "n_estimators": 2500, "learning_rate": 0.018, "max_depth": 4,
            "min_child_weight": 18, "subsample": 0.72, "colsample_bytree": 0.50,
            "colsample_bylevel": 0.80, "reg_alpha": 2.5, "reg_lambda": 10.0, "gamma": 0.40,
        },
    },
}

oof_map: Dict[str, np.ndarray] = {}
test_map: Dict[str, np.ndarray] = {}

print("Addestramento dei 4 modelli in corso...")
for m_name, m_spec in models_cfg.items():
    cols = subsets[m_spec["subset"]]
    cv_res, test_p = run_xgb_cv(
        X_df=X_train_full[cols],
        y=y,
        splits=cv_splits,
        xgb_params=m_spec["params"],
        experiment_id=m_name,
        early_stopping_rounds=50,
        fold_feature_fn=fold_te_fn if m_spec["use_te"] else None,
        X_test_df=X_test_full[cols],
        test_fold_feature_fn=test_te_fn if m_spec["use_te"] else None,
        threshold=DECISION_THRESHOLD,
        seed=RANDOM_SEED,
    )
    oof_map[m_name] = cv_res.oof_probs
    test_map[m_name] = test_p
    print(f"  [{m_name:36s}] CV Acc: {cv_res.cv_mean_acc:.4f} +/- {cv_res.cv_std_acc:.4f} | AUC: {cv_res.oof_roc_auc:.4f}")

## Step 5: Blending Probabilistico Ottimizzato (SLSQP con Floor e Shrinkage)
Ottimizziamo i pesi di voto con Sequential Least Squares Programming garantendo un diversity floor del 10% a Model 3 (Zero-TE hedge) e 20% di shrinkage verso pesi uniformi.

In [ ]:
from sklearn.metrics import accuracy_score, roc_auc_score, log_loss
from src.model import optimize_ensemble_weights

members = list(models_cfg.keys())
P_oof = np.column_stack([oof_map[m] for m in members])
P_test = np.column_stack([test_map[m] for m in members])

weights = optimize_ensemble_weights(
    y_true=y,
    oof_predictions=P_oof,
    init_weights=[0.15, 0.35, 0.10, 0.40],
    min_weight=0.10,
    shrinkage=0.20,
    metric="logloss",
)

blended_oof = np.dot(P_oof, weights)
blended_test = np.dot(P_test, weights)

oof_acc = accuracy_score(y, (blended_oof >= DECISION_THRESHOLD).astype(int))
oof_auc = roc_auc_score(y, blended_oof)
oof_ll = log_loss(y, blended_oof)

print("Pesi finali ensemble SLSQP:", [round(float(w), 4) for w in weights])
print("=" * 60)
print(f"OOF Ensemble Accuracy : {oof_acc:.4f}")
print(f"OOF Ensemble ROC-AUC  : {oof_auc:.4f}")
print(f"OOF Ensemble Log-Loss : {oof_ll:.4f}")
print("=" * 60)

## Step 6: Generazione del File Finale `submission.csv`
Scriviamo il file CSV finale applicando la soglia di Bayes $t=0.50$.

In [ ]:
from src.submission import generate_submission, validate_submission_df

output_csv_path = Path("submission.csv")
binary_preds = (blended_test >= DECISION_THRESHOLD).astype(int)

submission_df = generate_submission(
    test_battle_ids=test_ids,
    test_predictions=binary_preds,
    sample_sub_df=sample_sub,
    output_path=output_csv_path,
)

print(f"File generato: {output_csv_path.resolve()}")
display(submission_df.head(10))

## Step 7: Sanity Check e Validazione di Conformità al 100% per Kaggle
Eseguiamo la suite di controlli contrattuali formali e diagnostici per verificare che il file sia pronto all'invio.

In [ ]:
# 1. Validazione del contratto formale di competizione
sub_info = validate_submission_df(submission_df, sample_sub)

# 2. Verifiche di integrità del file su disco
assert output_csv_path.exists(), "ERRORE: submission.csv non trovato sul disco!"
file_size = output_csv_path.stat().st_size
assert file_size > 10000, f"ERRORE: dimensione file anomala ({file_size} bytes)"

# 3. Controllo formato testuale e assenza righe vuote
with open(output_csv_path, "r", encoding="utf-8") as f:
    lines = [line.strip() for line in f if line.strip()]

assert lines[0] == "battle_id,player_won", f"Header errato: {lines[0]}"
assert len(lines) == 5001, f"Numero righe (incluso header) errato: {len(lines)} (atteso 5001)"

# 4. Check valori e distribuzione delle classi
p1_wins = sub_info["positive_count"]
p2_wins = sub_info["negative_count"]
p1_rate = sub_info["positive_rate"]

assert 0.40 <= p1_rate <= 0.60, f"Bilanciamento anomalo: {p1_rate:.2%}"

# 5. Check anti-inquinamento della working directory di Kaggle
all_csvs = list(Path.cwd().glob("*.csv"))
kaggle_ready = len(all_csvs) == 1 and all_csvs[0].name == "submission.csv"

print("=" * 75)
print("CERTIFICAZIONE DI CONFORMITA' PER IL SITO KAGGLE: APPROVATO ")
print("=" * 75)
print(f"• Percorso file           : {output_csv_path.resolve()}")
print(f"• Dimensione su disco     : {file_size:,} bytes")
print(f"• Totale predizioni test  : {sub_info['n_rows']:,}")
print(f"• Valori Nulli / NaN      : 0")
print(f"• Tipo dato 'player_won'  : int64 (esclusivamente 0 o 1)")
print(f"• Vittorie Player 1 (1)   : {p1_wins:,} ({p1_rate*100:.2f}%)")
print(f"• Vittorie Player 2 (0)   : {p2_wins:,} ({(1-p1_rate)*100:.2f}%)")
print(f"• File CSV in working dir : {[p.name for p in all_csvs]} (Esattamente e solo submission.csv: {kaggle_ready})")
print("=" * 75)
print("\n>> Il file 'submission.csv' e' pronto per essere inviato con successo sulla piattaforma Kaggle!")